# QA 4 (Unit 4): Clustering Implementation and Visualization
## K-Means and DBSCAN on Customer Data
---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

# Set random seed for reproducibility
np.random.seed(42)

print('Libraries imported successfully!')

## Step 2: Generate Synthetic Customer Dataset
We simulate a customer dataset with **Annual Income** and **Spending Score** features.

In [ ]:
# Generate synthetic customer data with distinct clusters + noise
n_samples = 300

# 5 main customer segments
centers = [
    [20, 70],   # Low income, high spenders
    [60, 60],   # Mid income, mid-high spenders
    [90, 15],   # High income, low spenders
    [30, 20],   # Low income, low spenders
    [80, 85],   # High income, high spenders
]
X_main, _ = make_blobs(n_samples=n_samples, centers=centers, cluster_std=5.0)

# Add some noise/outlier points (simulating irregular customers)
noise = np.random.uniform(low=[0, 0], high=[100, 100], size=(20, 2))
X = np.vstack([X_main, noise])

# Clip values to [0, 100]
X[:, 0] = np.clip(X[:, 0], 0, 100)
X[:, 1] = np.clip(X[:, 1], 0, 100)

# Create DataFrame
df = pd.DataFrame(X, columns=['Annual Income (k$)', 'Spending Score (1-100)'])

print(f'Dataset shape: {df.shape}')
print(df.describe().round(2))

## Step 3: Data Preprocessing — Standardization

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print('Data scaled. Mean:', X_scaled.mean(axis=0).round(4))
print('Std Dev:', X_scaled.std(axis=0).round(4))

## Step 4: Elbow Method — Find Optimal K for K-Means

In [ ]:
inertias = []
silhouette_scores = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].axvline(x=5, color='red', linestyle='--', label='k=5 (chosen)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), silhouette_scores, 'go-', linewidth=2, markersize=7)
axes[1].set_title('Silhouette Score vs k', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=5, color='red', linestyle='--', label='k=5 (chosen)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Optimal k = 5 based on the elbow and high silhouette score.')

## Step 5: K-Means Clustering

In [ ]:
# Apply K-Means with k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10, max_iter=300)
kmeans_labels = kmeans.fit_predict(X_scaled)

df['KMeans_Cluster'] = kmeans_labels

# Silhouette score
km_silhouette = silhouette_score(X_scaled, kmeans_labels)
print(f'K-Means Silhouette Score: {km_silhouette:.4f}')
print(f'K-Means Inertia: {kmeans.inertia_:.2f}')
print('\nCluster sizes:')
print(pd.Series(kmeans_labels).value_counts().sort_index())

## Step 6: DBSCAN Clustering

In [ ]:
# Apply DBSCAN
dbscan = DBSCAN(eps=0.35, min_samples=8)
dbscan_labels = dbscan.fit_predict(X_scaled)

df['DBSCAN_Cluster'] = dbscan_labels

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f'DBSCAN found {n_clusters_db} clusters')
print(f'Noise/Outlier points: {n_noise}')

if n_clusters_db > 1:
    # Silhouette only for non-noise points
    mask = dbscan_labels != -1
    db_silhouette = silhouette_score(X_scaled[mask], dbscan_labels[mask])
    print(f'DBSCAN Silhouette Score (non-noise): {db_silhouette:.4f}')

print('\nCluster label counts (−1 = noise):')
print(pd.Series(dbscan_labels).value_counts().sort_index())

## Step 7: Visualization — Side-by-Side 2D Plots

In [ ]:
palette_km  = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']
palette_db  = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Clustering Comparison: K-Means vs DBSCAN', fontsize=16, fontweight='bold', y=1.01)

# --- K-Means Plot ---
ax = axes[0]
for cluster_id in sorted(df['KMeans_Cluster'].unique()):
    mask = df['KMeans_Cluster'] == cluster_id
    ax.scatter(
        df.loc[mask, 'Annual Income (k$)'],
        df.loc[mask, 'Spending Score (1-100)'],
        c=palette_km[cluster_id], label=f'Cluster {cluster_id+1}',
        s=60, alpha=0.85, edgecolors='white', linewidths=0.4
    )
# Plot centroids (inverse transform back to original scale)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
ax.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroids')
ax.set_title('K-Means Clustering (k=5)', fontsize=13, fontweight='bold')
ax.set_xlabel('Annual Income (k$)', fontsize=11)
ax.set_ylabel('Spending Score (1-100)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# --- DBSCAN Plot ---
ax2 = axes[1]
unique_labels = sorted(df['DBSCAN_Cluster'].unique())
for i, cluster_id in enumerate(unique_labels):
    mask = df['DBSCAN_Cluster'] == cluster_id
    if cluster_id == -1:
        ax2.scatter(
            df.loc[mask, 'Annual Income (k$)'],
            df.loc[mask, 'Spending Score (1-100)'],
            c='gray', marker='x', s=80, label='Noise', zorder=3, linewidths=1.5
        )
    else:
        ax2.scatter(
            df.loc[mask, 'Annual Income (k$)'],
            df.loc[mask, 'Spending Score (1-100)'],
            c=palette_db[cluster_id % len(palette_db)],
            label=f'Cluster {cluster_id+1}',
            s=60, alpha=0.85, edgecolors='white', linewidths=0.4
        )
ax2.set_title(f'DBSCAN Clustering (eps=0.35, min_samples=8)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Annual Income (k$)', fontsize=11)
ax2.set_ylabel('Spending Score (1-100)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('kmeans_vs_dbscan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as kmeans_vs_dbscan.png')

## Step 8: Cluster Analysis and Observations

In [ ]:
print('=' * 55)
print('       K-MEANS CLUSTER SUMMARY')
print('=' * 55)
km_summary = df.groupby('KMeans_Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
km_summary.index = [f'Cluster {i+1}' for i in km_summary.index]
print(km_summary)

print('\nCustomer Segment Interpretation:')
segments = [
    'Low Income, High Spenders  → "Impulsive Buyers"',
    'Mid Income, Mid Spenders   → "Standard Customers"',
    'High Income, Low Spenders  → "Careful / Conservative"',
    'Low Income, Low Spenders   → "Budget-Conscious"',
    'High Income, High Spenders → "VIP / Premium Target"',
]
for i, seg in enumerate(segments):
    print(f'  Cluster {i+1}: {seg}')

print()
print('=' * 55)
print('       DBSCAN CLUSTER SUMMARY')
print('=' * 55)
db_summary = df[df['DBSCAN_Cluster'] != -1].groupby('DBSCAN_Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
print(db_summary)
print(f'\nNoise points (outliers): {(df["DBSCAN_Cluster"] == -1).sum()}')

## Step 9: Algorithm Comparison Summary

In [ ]:
comparison = pd.DataFrame({
    'Property': [
        'Number of clusters',
        'Noise handling',
        'Shape assumption',
        'Need to specify k/params',
        'Silhouette Score',
        'Sensitive to outliers',
        'Best for',
    ],
    'K-Means': [
        '5 (user-defined)',
        'No (assigns all points)',
        'Spherical / convex',
        'Yes — k must be specified',
        f'{km_silhouette:.4f}',
        'Yes — pulls centroids',
        'Well-separated, round clusters',
    ],
    'DBSCAN': [
        f'{n_clusters_db} (auto-detected)',
        'Yes (labels as −1)',
        'Arbitrary shapes',
        'Yes — eps & min_samples',
        f'{db_silhouette:.4f} (non-noise)',
        'No — noise is excluded',
        'Irregular shapes, noisy data',
    ]
})

print(comparison.to_string(index=False))

print()
print('CONCLUSION')
print('-' * 50)
print('K-Means performs better on this dataset because the')
print('customer segments are approximately spherical and')
print('well-separated. K-Means produces clear boundaries')
print('and assigns all points to a cluster.')
print()
print('DBSCAN excels at detecting outliers and arbitrary')
print('shapes — it correctly flags sparse noise points that')
print('K-Means would incorrectly absorb into a cluster.')